## Trabalho 1 da Disciplina UFSM00993 - Aprendizado de Máquina Profundo

### Você deverá implementar, usando exclusivamente NumPy e Matplotlib, uma Multi-layer Perceptron (MLP) para classificação de dados bidimensionais. O objetivo é compreender como a arquitetura da rede, as funções de ativação e o processo de otimização influenciam a capacidade de aprendizado de um modelo de aprendizado profundo.

#### 1. Considere o dataset fornecido em formato .json. Separe os dados adequadamente em subconjuntos de treino e validação. Considere normalizar os dados de treino e validação conforme estatísticas (média e desvio padrão) dos dados de treino de tal forma que os dados de treino normalizados tenham média zero e desvio padrao unitário. Em todos seus testes, atente para potenciais sobreajustes. Utilize uma seed fixa para separação dos dados e inicializações dos parâmetros dos modelos.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import json

In [2]:
rng = np.random.default_rng(7)  # seed fixa para ficar reproduzivel

In [3]:
print(f'número do dataset usado: {202510242 % 8}')

número do dataset usado: 2


In [4]:
# !wget https://raw.githubusercontent.com/tltsilveira/ufsm00993/main/dataset_2.json comentado só para não ficar baixando toda hora

In [5]:
with open("dataset_2.json", "r") as file:
    dataset = json.load(file)

X = np.array(dataset["X"], dtype=float)
y = np.array(dataset["y"], dtype=int)

In [6]:
X.shape, y.shape

((600, 2), (600,))

Com 600 dados, vou dividir agora em uma proporção 80/20, 80% dos dados para treino e 20% para validação. Depois, computo a média e desvio padrão somento no conjunto de treino para normalizar os conjuntos, evitando data leakage.

In [7]:
600 * 0.8

480.0

In [8]:
indexes = rng.permutation(len(X)) # permutação dos indices para separar aleatoriamente train e val

index_train = indexes[:480]
index_val = indexes[480:]

X_train = X[index_train]
y_train = y[index_train]
X_val = X[index_val]
y_val = y[index_val]

In [9]:
X_train.shape, y_train.shape, X_val.shape, y_val.shape

((480, 2), (480,), (120, 2), (120,))

In [10]:
y_train.sum() / len(y_train) # 51% dos dados de treino é classe 1

np.float64(0.5145833333333333)

In [11]:
y_val.sum() / len(y_val) # 45% dos dados de validacao é classe 1, tá razoavelmente estratificado

np.float64(0.4583333333333333)

agora, normalizar os dois conjuntos usando as estatísticas do conjunto de treino

In [12]:
mean_train = X_train.mean(axis=0)
std_train = X_train.std(axis=0)

mean_train, std_train

(array([ 0.06034041, -0.00356361]), array([1.74915666, 1.13727365]))

In [13]:
X_train_norm = (X_train - mean_train) / std_train
X_val_norm = (X_val - mean_train) / std_train

X_train_norm.shape, X_val_norm.shape

((480, 2), (120, 2))

In [14]:
X_train_norm.mean(axis=0), X_train_norm.std(axis=0)

(array([5.55111512e-17, 1.85037171e-17]), array([1., 1.]))

In [15]:
X_val_norm.mean(axis=0), X_val_norm.std(axis=0)

(array([-0.15553578,  0.03440187]), array([1.03422946, 1.06103242]))

#### 2. Implemente, utilizando apenas Python e NumPy, uma MLP com duas entradas, uma camada oculta com largura a ser escolhida e uma camada de saída adequada ao problema de classificação considerado. Cuide para que sua camada oculta tenha uma função de ativacao não-linear.

arquitetura inicial pensada: 2 units de entrada, 8 units na camada oculta, 1 unit de saida. na camada oculta uso da fun,ão de ativação relu e na de saída sigmoid.

y_hat = sig(relu(X @ W1 + b1) @ W2 + b2)

In [16]:
# funções de ativação usadas
def relu(x):
    return np.maximum(0, x)

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

In [17]:
W1 = rng.normal(0, 1, size=(2, 8))
b1 = np.zeros((1, 8))
W2 = rng.normal(0, 1, size=(8, 1))
b2 = np.zeros((1, 1))


W1, b2, W2, b1

(array([[-0.97714544,  0.85343765, -0.51816969,  1.49830168, -0.77983936,
          0.38650199, -0.22728301, -0.75402181],
        [ 0.58767504, -0.15498257,  0.60321058, -0.04729151, -1.08581626,
         -0.10206924,  0.05195517,  0.95846295]]),
 array([[0.]]),
 array([[-0.90626901],
        [-0.03932985],
        [-1.72187873],
        [ 0.65149306],
        [-1.08148297],
        [-1.80635952],
        [-0.05921683],
        [ 1.10568496]]),
 array([[0., 0., 0., 0., 0., 0., 0., 0.]]))

In [18]:
def init_params(rng):
    W1 = rng.normal(0, 1, size=(2, 8))
    b1 = np.zeros((1, 8))
    W2 = rng.normal(0, 1, size=(8, 1))
    b2 = np.zeros((1, 1))
    return W1, b1, W2, b2


def forward(X, W1, b1, W2, b2):
    A1 = relu(X @ W1 + b1)
    y_hat = sigmoid(A1 @ W2 + b2)
    return y_hat

In [19]:
W1, b1, W2, b2 = init_params(rng)
y_hat = forward(X_train_norm, W1, b1, W2, b2)

In [20]:
y_hat.shape

(480, 1)

In [21]:
y_hat[:5]

array([[0.90711444],
       [0.00822699],
       [0.22376671],
       [0.01975817],
       [0.15623655]])

#### 3. Implemente manualmente uma função de perda adequada ao problema e suas respectivas funções de ativação na camada de saída. Considere e discuta as possibilidades de utilizar binary cross-entropy (BCE) em conjunto com a função sigmoide ou categorical cross-entropy (CCE) em conjunto com a função softmax.

para usar a saida sigmoid e binary cross-entropy se utiliza somente um neuronio de saida, com valores entre 0 e 1. já para usar a softmax e CCE seriam necessarios 2 neuronios, um para cada classe (0 e 1). como estou usando 1 neuronio para saida, vou seguir usando a BCE

In [22]:
def bce(y, y_hat):
    y_hat = y_hat.ravel() # deixar no formato vetor assim como y, (480, 1) -> (480,)
    y_hat = np.clip(y_hat, a_min=1e-12, a_max=1 - 1e-12) # evitar logaritmo indefinido log(0) ou log(1-1)
    return -np.mean((1 - y) * np.log(1 - y_hat) + y * np.log(y_hat))

In [23]:
bce(y_train, y_hat)

np.float64(1.4632724794983163)

#### 4. Inicialize os pesos utilizando a He initialization. Para uma camada com nin entradas, os pesos devem ser amostrados de uma distribuição normal com média zero e desvio-padrão σ = sqrt(2 / nin). Os vieses deverão ser inicializados com zero.

In [24]:
def he_init_params(rng):
    W1 = rng.normal(0, np.sqrt(2 / 2), size=(2, 8))
    b1 = np.zeros((1, 8))
    W2 = rng.normal(0, np.sqrt(2 / 8), size=(8, 1))
    b2 = np.zeros((1, 1))
    return W1, b1, W2, b2

In [25]:
W1, b1, W2, b2 = he_init_params(rng)
y_hat = forward(X_train_norm, W1, b1, W2, b2)

In [26]:
bce(y_train, y_hat)

np.float64(0.9811993763307262)